# 30 — Validate `usdm_v4.ttl`

Parses the generated Turtle, runs SPARQL sanity counts, cross-checks NCIt
anchors against the source YAML, and appends a CSV row to
`../reports/validation.csv`.

Comparison against the prototype baseline (8,215 triples) is informational.
Deviation flags either source drift (likely benign, document in `docs/`) or
a generation bug (investigate).


## 1. Parse `usdm_v4.ttl`

Fail fast on syntax errors. Successful parse ⇒ the file is valid Turtle.


In [1]:
import rdflib
from pathlib import Path

TTL_PATH = "../usdm_v4.ttl"
if not Path(TTL_PATH).exists():
    raise FileNotFoundError(
        f"{TTL_PATH} missing — run notebooks/20_generate_turtle.ipynb first."
    )

g = rdflib.Graph()
g.parse(TTL_PATH, format="turtle")
print(f"parsed {len(g)} triples from {TTL_PATH}")


parsed 8069 triples from ../usdm_v4.ttl


## 2. Load source YAML for cross-checks

We re-parse `dataStructure.yml` here to compare what's in the graph
against what's in source.


In [2]:
import yaml
import json

YAML_PATH = "../downloads/dataStructure.yml"
META_PATH = "../downloads/.fetch_meta.json"

with open(YAML_PATH) as f:
    SOURCE = yaml.safe_load(f)
with open(META_PATH) as f:
    META = json.load(f)

print(f"YAML  : {len(SOURCE)} top-level entries")
print(f"tag   : {META['ddf_ra_tag']}")
print(f"sha256: {META['sha256']}")


YAML  : 86 top-level entries
tag   : v4.0.0
sha256: 6f49a407aef41a66dd40a82ad3348672a22cd8874849254d7e4f27d6275daca6


## 3. SPARQL sanity counts

Named classes (filter to IRIs to exclude `owl:unionOf` blank nodes).
Properties = `owl:DatatypeProperty` ∪ `owl:ObjectProperty`.


In [3]:
QUERIES = {
    "triples": (
        "SELECT (COUNT(*) AS ?n) WHERE { ?s ?p ?o }"
    ),
    "classes_total": (
        "SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?c a owl:Class . FILTER(isIRI(?c)) }"
    ),
    "classes_nci_anchored": (
        "PREFIX skos:<http://www.w3.org/2004/02/skos/core#> "
        "SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { "
        "  ?c a owl:Class ; skos:exactMatch ?x . FILTER(isIRI(?c)) }"
    ),
    "properties_total": (
        "SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE { "
        "  { ?p a owl:DatatypeProperty } UNION { ?p a owl:ObjectProperty } }"
    ),
    "properties_nci_anchored": (
        "PREFIX skos:<http://www.w3.org/2004/02/skos/core#> "
        "SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE { "
        "  ?p skos:exactMatch ?x . "
        "  FILTER EXISTS { { ?p a owl:DatatypeProperty } UNION { ?p a owl:ObjectProperty } } }"
    ),
    "value_props": (
        "PREFIX usdm:<https://example.org/cdisc/usdm/v4/> "
        "SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE { ?p usdm:relationshipType 'Value' }"
    ),
    "ref_props": (
        "PREFIX usdm:<https://example.org/cdisc/usdm/v4/> "
        "SELECT (COUNT(DISTINCT ?p) AS ?n) WHERE { ?p usdm:relationshipType 'Ref' }"
    ),
    "concrete_classes": (
        "PREFIX usdm:<https://example.org/cdisc/usdm/v4/> "
        "SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?c usdm:modifier 'Concrete' . FILTER(isIRI(?c)) }"
    ),
    "abstract_classes": (
        "PREFIX usdm:<https://example.org/cdisc/usdm/v4/> "
        "SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?c usdm:modifier 'Abstract' . FILTER(isIRI(?c)) }"
    ),
}

counts = {}
for label, sparql in QUERIES.items():
    counts[label] = int(list(g.query(sparql))[0][0])
    print(f"  {label:<25}: {counts[label]}")


  triples                  : 8069
  classes_total            : 86
  classes_nci_anchored     : 84
  properties_total         : 693
  properties_nci_anchored  : 312
  value_props              : 630
  ref_props                : 63
  concrete_classes         : 80
  abstract_classes         : 6


## 4. Cross-check class-level NCIt anchors

For every source class with `NCI C-Code`, the graph must contain
`usdm:{Class} skos:exactMatch ncit:{C-Code}`. Mismatches are collected,
not raised — we want a complete report, not a first-failure exit.


In [4]:
USDM = rdflib.Namespace("https://example.org/cdisc/usdm/v4/")
NCIT = rdflib.Namespace("http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#")
SKOS = rdflib.Namespace("http://www.w3.org/2004/02/skos/core#")

class_mismatches = []
for cname, entry in SOURCE.items():
    if not isinstance(entry, dict) or "NCI C-Code" not in entry:
        continue
    expected = NCIT[entry["NCI C-Code"]]
    subject = USDM[cname]
    if (subject, SKOS.exactMatch, expected) not in g:
        class_mismatches.append((cname, entry["NCI C-Code"]))

print(f"class NCIt cross-check: {len(class_mismatches)} mismatches")
for c, code in class_mismatches[:20]:
    print(f"  MISSING: usdm:{c} skos:exactMatch ncit:{code}")


class NCIt cross-check: 0 mismatches


## 5. Cross-check attribute-level NCIt anchors

Same check, scoped to non-inherited attributes. Property IRI is
`usdm:{ClassName}-{attributeName}`.


In [5]:
attr_mismatches = []
for cname, entry in SOURCE.items():
    if not isinstance(entry, dict):
        continue
    for aname, attr in (entry.get("Attributes") or {}).items():
        if not isinstance(attr, dict) or "Inherited From" in attr:
            continue
        if "NCI C-Code" not in attr:
            continue
        expected = NCIT[attr["NCI C-Code"]]
        subject = USDM[f"{cname}-{aname}"]
        if (subject, SKOS.exactMatch, expected) not in g:
            attr_mismatches.append((cname, aname, attr["NCI C-Code"]))

print(f"attribute NCIt cross-check: {len(attr_mismatches)} mismatches")
for c, a, code in attr_mismatches[:20]:
    print(f"  MISSING: usdm:{c}-{a} skos:exactMatch ncit:{code}")


attribute NCIt cross-check: 0 mismatches


## 6. Compare against prototype baseline

Baseline numbers come from prior prototype work against an earlier DDF-RA
snapshot. The triple-count baseline (8,215) reflects choices in that
prototype we cannot fully reconstruct — a residual delta of <300 is
expected and documented in `docs/known-gaps.md`. Structural counts
(classes, properties, anchored counts) should match exactly for tag
`v4.0.0` of DDF-RA.


In [6]:
BASELINE = {
    "triples":                 8215,
    "classes_total":             86,
    "classes_nci_anchored":      84,
    "properties_total":         693,
    "properties_nci_anchored":  312,
}

print(f"{'metric':<25} {'baseline':>10} {'observed':>10} {'delta':>10}")
print("-" * 60)
for k, base in BASELINE.items():
    obs = counts[k]
    delta = obs - base
    print(f"{k:<25} {base:>10} {obs:>10} {delta:>+10}")


metric                      baseline   observed      delta
------------------------------------------------------------
triples                         8215       8069       -146
classes_total                     86         86         +0
classes_nci_anchored              84         84         +0
properties_total                 693        693         +0
properties_nci_anchored          312        312         +0


## 7. Append CSV row to `../reports/validation.csv`

One row per validation run, header on first write. History accumulates
in the file across runs.


In [7]:
import csv
import datetime
from pathlib import Path

REPORT_PATH = "../reports/validation.csv"

row = {
    "timestamp_utc":           datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "ddf_ra_tag":              META["ddf_ra_tag"],
    "source_sha256":           META["sha256"],
    "triples":                 counts["triples"],
    "classes_total":           counts["classes_total"],
    "classes_nci_anchored":    counts["classes_nci_anchored"],
    "properties_total":        counts["properties_total"],
    "properties_nci_anchored": counts["properties_nci_anchored"],
    "value_props":             counts["value_props"],
    "ref_props":               counts["ref_props"],
    "concrete_classes":        counts["concrete_classes"],
    "abstract_classes":        counts["abstract_classes"],
    "class_mismatches":        len(class_mismatches),
    "attr_mismatches":         len(attr_mismatches),
}

Path(REPORT_PATH).parent.mkdir(parents=True, exist_ok=True)
first_write = not Path(REPORT_PATH).exists()
with open(REPORT_PATH, "a", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(row.keys()))
    if first_write:
        w.writeheader()
    w.writerow(row)

action = "created" if first_write else "appended to"
print(f"{action} {REPORT_PATH}")
for k, v in row.items():
    print(f"  {k:<25}: {v}")


created ../reports/validation.csv
  timestamp_utc            : 2026-04-26T07:33:36.500555+00:00
  ddf_ra_tag               : v4.0.0
  source_sha256            : 6f49a407aef41a66dd40a82ad3348672a22cd8874849254d7e4f27d6275daca6
  triples                  : 8069
  classes_total            : 86
  classes_nci_anchored     : 84
  properties_total         : 693
  properties_nci_anchored  : 312
  value_props              : 630
  ref_props                : 63
  concrete_classes         : 80
  abstract_classes         : 6
  class_mismatches         : 0
  attr_mismatches          : 0


## 8. Run summary

Empty mismatch lists ⇒ every NCI C-Code in source has a matching
`skos:exactMatch` triple in the graph. Non-empty lists are real bugs
to investigate.


In [8]:
n_class_mm = len(class_mismatches)
n_attr_mm = len(attr_mismatches)

if n_class_mm == 0 and n_attr_mm == 0:
    print("clean run: all source NCI anchors present in graph")
else:
    print(f"class mismatches:     {n_class_mm}")
    print(f"attribute mismatches: {n_attr_mm}")
    print("see cells 4 and 5 for details")


clean run: all source NCI anchors present in graph
